# Liu2024 Source `.mat` SignalJEPA PreLocal Fine-Tuning

This notebook is the Liu2024-only S-JEPA downstream notebook.

It intentionally **does not use MOABB windowing** for Liu2024. Instead, it loads the original Figshare source `.mat` files in the same style as the paper-mimic/tuning notebook, then applies the S-JEPA-style preprocessing and trains **only SignalJEPA PreLocal**.

Main design choices:

- source loading: original Figshare `sourcedata` `.mat` files
- preprocessing: EEG-only, average reference, resample to 128 Hz, 0.5–40 Hz filter
- window: fixed 537 samples, starting at 2.0 s inside each original 8 s source trial
- downstream model: `SignalJEPA_PreLocal` only
- evaluation: within-subject stratified 5-fold CV

# 1. Setup

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import re
import sys
import json
import math
import hashlib
import random
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset, Subset

from scipy.io import loadmat

import mne
mne.set_log_level("WARNING")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal

print("Runtime Environment:")
print(f"  Python:      {sys.version}")
print(f"  Platform:    {platform.platform()}")
print(f"  PyTorch:     {torch.__version__}")
print(f"  MNE:         {mne.__version__}")
print(f"  Working dir: {Path.cwd()}")

/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Runtime Environment:
  Python:      3.11.15 (main, Apr  9 2026, 01:18:52) [Clang 21.0.0 (clang-2100.0.123.102)]
  Platform:    macOS-26.2-arm64-arm-64bit
  PyTorch:     2.10.0
  MNE:         1.11.0
  Working dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src


# 2. Configuration

## 2.1 CONFIG

In [2]:
WORKING_DIR = Path.cwd()

CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-prelocal"),
    # Optional manual override. Leave as None to auto-detect.
    # Expected to point somewhere under extracted Figshare sourcedata.
    "manual_source_extract_dir": None,

    # Dataset
    "dataset_name": "Liu2024_SourceMAT",
    "labels_to_keep": ["left_hand", "right_hand"],
    "subjects_to_use": None,  # None = all available source .mat subjects
    "exclude_subjects": [],

    # Source data conventions
    "source_sfreq": 500,
    "source_trial_duration_s": 8.0,
    # 0..29 are EEG-like channels, source index 17 is CPz/reference,
    # 30..31 are EOG, 32 is marker. This matches the previous source-mat notebook.
    "drop_source_reference_channel": True,
    "drop_eog_and_marker": True,
    # Source .mat data appears already in EEG amplitude units used by the original scripts.
    # Do not multiply by 1e6 unless you explicitly confirm the data are in volts.
    "source_data_scale": 1.0,

    # S-JEPA-style preprocessing
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,
    # This mirrors the current MOABB S-JEPA notebook order more closely.
    "average_reference_before_resample_filter": True,

    # S-JEPA-compatible downstream window
    "target_window_samples": 537,
    "mi_window_start_s": 2.0,

    # Model settings
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",  # from_pretrained or random
    # This follows the old Liu2024 MOABB notebook, which used the without-chans checkpoint
    # for mismatched channel sets.
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",

    # Fine-tuning strategy
    "strategy": "new",  # new or full
    "warmup_epochs": 10,

    # Cross-validation
    "cv_folds": 5,
    "val_split": 0.2,

    # Training
    "batch_size": 32,
    "n_epochs": 50,
    "early_stopping_patience": None,
    "learning_rate": 0.001,

    # Reproducibility
    "seed": 12,
    "set_seed": True,

    # Diagnostics
    "extract_spatial_conv_weights": True,
    "collapse_threshold": 0.90,
    "log_spatial_update_stats": True,
    "log_probability_diagnostics": True,
}

# Fixed Liu source channel map, using modern 10-20 aliases where appropriate.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17  # CPz in the paper/source channel table.
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T7", "T8", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "P7", "P8", "Oz", "O1", "O2",
]

if CONFIG["drop_source_reference_channel"]:
    EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
    EEG_CHANNEL_NAMES = [name for i, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30) if i != SOURCE_REFERENCE_INDEX]
else:
    EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_30
    EEG_CHANNEL_NAMES = SOURCE_EEG_CHANNEL_NAMES_30

TARGET_N_CLASSES = len(CONFIG["labels_to_keep"])
WINDOW_SAMPLES = int(CONFIG["target_window_samples"])
TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / float(CONFIG["sfreq"])
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG["mi_window_start_s"]) * float(CONFIG["sfreq"])))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES

print("Core configuration:")
print(f"  Dataset:                   {CONFIG['dataset_name']}")
print(f"  Model:                     {CONFIG['model_name']}")
print(f"  Pretrained mode:           {CONFIG['pretrained_mode']}")
print(f"  Pretrained repo:           {CONFIG['pretrained_repo_id']}")
print(f"  Source sfreq:              {CONFIG['source_sfreq']} Hz")
print(f"  Target sfreq:              {CONFIG['sfreq']} Hz")
print(f"  Target window:             {WINDOW_SAMPLES} samples = {TARGET_TRIAL_DURATION_S:.6f} s")
print(f"  Crop in resampled trial:   {MI_WINDOW_START_SAMPLE}:{MI_WINDOW_STOP_SAMPLE}")
print(f"  EEG channels used:         {len(EEG_CHANNEL_NAMES)}")

Core configuration:
  Dataset:                   Liu2024_SourceMAT
  Model:                     SignalJEPA_PreLocal
  Pretrained mode:           from_pretrained
  Pretrained repo:           braindecode/signal-jepa_without-chans
  Source sfreq:              500 Hz
  Target sfreq:              128 Hz
  Target window:             537 samples = 4.195312 s
  Crop in resampled trial:   256:793
  EEG channels used:         29


## 2.2 Artifact Creation and Logging Init

In [3]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / CONFIG["dataset_name"] / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass
    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    file = kwargs.pop("file", None)
    flush = kwargs.pop("flush", False)
    msg = sep.join(str(a) for a in args)
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}{end}"
    _safe_write_text(_LOG_FILE_HANDLE, line)
    _LOG_FILE_HANDLE.flush()
    if file is None:
        _safe_write_text(sys.__stdout__, line)
        if flush:
            sys.__stdout__.flush()
    else:
        _safe_write_text(file, line)
        if flush:
            file.flush()

import builtins
builtins.print = _timestamped_print

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Log path:   {LOG_PATH}")

with open(ARTIFACT_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

[2026-05-18 20:08:20] Run ID:     20260518_2008_8d0bbab1
[2026-05-18 20:08:20] Artifacts:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1
[2026-05-18 20:08:20] Log path:   /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/run.log


## 2.3 Reproducibility

In [4]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"]) if CONFIG["seed"] is not None else None
if CONFIG["set_seed"]:
    if BASE_SEED is None:
        raise ValueError("Seed control is enabled but CONFIG['seed'] is None.")
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")
else:
    print("Seed control disabled.")

[2026-05-18 20:08:20] Using device: mps
[2026-05-18 20:08:21] Seed initialized: 12


# 3. Load and Prepare Data

## 3.1 Locate Liu2024 Source `.mat` Files

In [5]:
def find_source_mat_files(root: Path):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(root.rglob("*.mat"))

def candidate_source_dirs():
    candidates = []
    if CONFIG["manual_source_extract_dir"] is not None:
        candidates.append(Path(CONFIG["manual_source_extract_dir"]))

    for base in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        candidates.extend([
            base / "liu2024_figshare" / "sourcedata",
            base / "liu2024_figshare" / "sourcedata" / "sourcedata",
            base / "src" / "liu2024_figshare" / "sourcedata",
            base / "src" / "liu2024_figshare" / "sourcedata" / "sourcedata",
        ])

    seen, uniq = set(), []
    for c in candidates:
        key = str(c.resolve()) if c.exists() else str(c)
        if key not in seen:
            uniq.append(c)
            seen.add(key)
    return uniq

MAT_FILES = []
SOURCE_EXTRACT_DIR = None
for cand in candidate_source_dirs():
    files = find_source_mat_files(cand)
    if files:
        MAT_FILES = files
        SOURCE_EXTRACT_DIR = cand
        break

if not MAT_FILES:
    print("Could not auto-find Liu2024 source .mat files.")
    print("Checked these candidate directories:")
    for c in candidate_source_dirs():
        print(f"  {c}")
    raise FileNotFoundError("Set CONFIG['manual_source_extract_dir'] to your extracted Figshare sourcedata directory.")

print(f"Source extract dir: {SOURCE_EXTRACT_DIR}")
print(f"Found {len(MAT_FILES)} .mat files")
print("First 5 files:")
for p in MAT_FILES[:5]:
    print(f"  {p}")

[2026-05-18 20:08:21] Source extract dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata
[2026-05-18 20:08:21] Found 50 .mat files
[2026-05-18 20:08:21] First 5 files:
[2026-05-18 20:08:21]   /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-01/sub-01_task-motor-imagery_eeg.mat
[2026-05-18 20:08:21]   /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-02/sub-02_task-motor-imagery_eeg.mat
[2026-05-18 20:08:21]   /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-03/sub-03_task-motor-imagery_eeg.mat
[2026-05-18 20:08:21]   /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-04/sub-04_task-motor-imagery_eeg.mat
[2026-05-18 20:08:21]   /Users

## 3.2 Robust Source `.mat` Loader

In [6]:
def subject_id_from_path(path: Path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Yield (name, value) recursively from scipy-loaded MATLAB structs."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                name = f"{prefix}{idx}"
                yield from _walk_mat_object(item, name)

def mat_structure_preview(path: Path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({"name": name, "type": "ndarray", "shape": str(value.shape), "dtype": str(value.dtype)})
        else:
            rows.append({"name": name, "type": type(value).__name__, "shape": "", "dtype": ""})
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata):
    """Return rawdata as trials x channels x samples."""
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Expected source shape is usually 40 x 33 x 4000.
    if arr.shape[0] in (39, 40) and arr.shape[1] >= 30 and arr.shape[2] >= 1000:
        return arr

    # Move trial axis to front.
    trial_axes = [i for i, s in enumerate(arr.shape) if s in (39, 40)]
    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # Move time axis to last.
    time_axis = int(np.argmax(arr.shape))
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    # Remaining middle axis should be channels.
    if arr.shape[1] < 30:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def load_subject_mat(path: Path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates = []
    label_candidates = []
    for name, arr in arrays:
        lname = name.lower()
        if arr.ndim == 3:
            score = 0
            if "rawdata" in lname or "data" in lname:
                score += 10
            if 39 <= min(arr.shape) <= 40 or arr.shape[0] in (39, 40):
                score += 3
            if max(arr.shape) >= 3000:
                score += 2
            raw_candidates.append((score, name, arr))
        elif arr.ndim in (1, 2):
            flat = arr.ravel()
            unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
            score = 0
            if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
                score += 10
            if flat.size in (39, 40):
                score += 3
            if unique and unique.issubset({"0", "1", "2"}):
                score += 2
            label_candidates.append((score, name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        print(f"MAT structure preview for failure at {path}:")
        print(preview.to_string(index=False))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    raw_score, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    label_score, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    rawdata = _normalize_rawdata_shape(raw_arr)
    labels = np.asarray(label_arr).astype(int).ravel()

    if len(labels) != rawdata.shape[0]:
        raise ValueError(
            f"Label count does not match trial count for {path}: labels={len(labels)}, rawdata={rawdata.shape}"
        )

    return rawdata, labels, raw_name, label_name

## 3.3 Verify Source Subjects

In [7]:
# Save a structure preview for the first subject to make debugging easy.
preview_path = ARTIFACT_DIR / "mat_structure_subject_01.csv"
mat_structure_preview(MAT_FILES[0]).to_csv(preview_path, index=False)
print(f"Wrote MAT structure preview: {preview_path}")

subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if CONFIG["subjects_to_use"] is not None and sid not in set(int(s) for s in CONFIG["subjects_to_use"]):
        continue
    if sid in set(int(s) for s in CONFIG["exclude_subjects"]):
        continue

    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    subjects.append({
        "subject_id": sid,
        "path": str(p),
        "rawdata_shape": tuple(X_raw.shape),
        "labels_shape": tuple(y_raw.shape),
        "label_counts_raw": np.bincount(y_raw.astype(int), minlength=3).tolist(),
        "raw_field": raw_field,
        "label_field": label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values("subject_id").reset_index(drop=True)
if subjects_df.empty:
    raise RuntimeError("No subjects were loaded after subject filtering.")

subjects_df.to_csv(ARTIFACT_DIR / "source_mat_summary.csv", index=False)
SUBJECTS = [int(s) for s in subjects_df["subject_id"].tolist()]

print(f"Subjects loaded: {SUBJECTS}")
print(subjects_df[["subject_id", "rawdata_shape", "labels_shape", "label_counts_raw", "raw_field", "label_field"]].head().to_string(index=False))

[2026-05-18 20:08:21] Wrote MAT structure preview: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/mat_structure_subject_01.csv
[2026-05-18 20:08:27] Subjects loaded: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
[2026-05-18 20:08:27]  subject_id  rawdata_shape labels_shape label_counts_raw   raw_field label_field
          1 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label
          2 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label
          3 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label
          4 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label
          5 (40, 33, 4000)        (40,)      [0, 20, 20] eeg.rawdata   eeg.label


## 3.4 S-JEPA-style Preprocessing from Source Trials

In [8]:
def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(EEG_CHANNEL_NAMES),
    )
    try:
        montage = mne.channels.make_standard_montage("standard_1020")
        info.set_montage(montage, match_case=False, on_missing="ignore")
    except Exception as exc:
        print(f"WARNING: Could not set standard_1020 montage: {exc}")
    return info

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def preprocess_subject_sjepa_style(rawdata, labels, subject_id):
    """
    Convert one source subject into S-JEPA-ready windows.

    rawdata expected: trials x 33 channels x 4000 samples at 500 Hz.
    output: trials x n_eeg_channels x 537 samples at 128 Hz.
    """
    if rawdata.ndim != 3:
        raise ValueError(f"Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}")
    if rawdata.shape[1] <= max(EEG_CHANNEL_INDICES):
        raise ValueError(f"Subject {subject_id}: not enough channels for selected indices; rawdata={rawdata.shape}")

    n_trials = rawdata.shape[0]
    X_eeg = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    X_eeg *= float(CONFIG.get("source_data_scale", 1.0))

    # Concatenate source trials into a per-subject RawArray, matching the earlier source notebook.
    # This mirrors continuous preprocessing style while still preserving the original 40 source trial labels.
    continuous = X_eeg.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    info = make_liu_info(CONFIG["source_sfreq"])
    raw = mne.io.RawArray(continuous, info, verbose=False)

    if CONFIG["average_reference_before_resample_filter"]:
        raw.set_eeg_reference("average", projection=False, verbose=False)

    raw.resample(float(CONFIG["sfreq"]), verbose=False)
    raw.filter(
        float(CONFIG["bandpass_low"]),
        float(CONFIG["bandpass_high"]),
        verbose=False,
    )

    if not CONFIG["average_reference_before_resample_filter"]:
        raw.set_eeg_reference("average", projection=False, verbose=False)

    data = raw.get_data()
    expected_samples_per_trial = int(round(rawdata.shape[2] * float(CONFIG["sfreq"]) / float(CONFIG["source_sfreq"])))
    total_expected = n_trials * expected_samples_per_trial

    if data.shape[1] != total_expected:
        n_full = data.shape[1] // n_trials
        print(
            f"WARNING subject {subject_id}: resampled samples {data.shape[1]} != expected {total_expected}; "
            f"using {n_full} samples/trial."
        )
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, expected_samples_per_trial).transpose(1, 0, 2)

    if MI_WINDOW_STOP_SAMPLE > X_rs.shape[-1]:
        raise ValueError(
            f"Subject {subject_id}: crop {MI_WINDOW_START_SAMPLE}:{MI_WINDOW_STOP_SAMPLE} exceeds "
            f"resampled trial length {X_rs.shape[-1]}"
        )

    X_win = X_rs[:, :, MI_WINDOW_START_SAMPLE:MI_WINDOW_STOP_SAMPLE]
    y = labels_to_zero_based(labels)

    if X_win.shape[-1] != WINDOW_SAMPLES:
        raise RuntimeError(f"Subject {subject_id}: expected {WINDOW_SAMPLES} samples, got {X_win.shape[-1]}")
    if len(y) != len(X_win):
        raise RuntimeError(f"Subject {subject_id}: labels/windows mismatch: {len(y)} vs {len(X_win)}")

    return X_win.astype(np.float32), y.astype(np.int64), int(expected_samples_per_trial)

# Build info after preprocessing target sfreq.
EEG_INFO = make_liu_info(CONFIG["sfreq"])
CHS_INFO = EEG_INFO["chs"]
CH_NAMES = list(EEG_CHANNEL_NAMES)

Xs, ys, subject_ids = [], [], []
window_summary_rows = []

for item in subjects_df.to_dict("records"):
    sid = int(item["subject_id"])
    X_raw, y_raw, raw_field, label_field = load_subject_mat(Path(item["path"]))
    X_win, y, samples_per_trial = preprocess_subject_sjepa_style(X_raw, y_raw, sid)

    Xs.append(X_win)
    ys.append(y)
    subject_ids.extend([sid] * len(y))

    window_summary_rows.append({
        "subject_id": sid,
        "n_windows": int(len(y)),
        "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist(),
        "window_shape": tuple(X_win.shape),
        "resampled_samples_per_trial": int(samples_per_trial),
        "crop_start_sample": int(MI_WINDOW_START_SAMPLE),
        "crop_stop_sample": int(MI_WINDOW_STOP_SAMPLE),
        "target_window_samples": int(WINDOW_SAMPLES),
        "effective_window_duration_s": float(TARGET_TRIAL_DURATION_S),
    })

X_ALL = np.concatenate(Xs, axis=0)
Y_ALL = np.concatenate(ys, axis=0)
SUBJECT_ID_ALL = np.asarray(subject_ids)

window_summary_df = pd.DataFrame(window_summary_rows).sort_values("subject_id")
window_summary_df.to_csv(ARTIFACT_DIR / "window_counts_by_subject.csv", index=False)

print(f"X_ALL shape:       {X_ALL.shape}")
print(f"Y_ALL class counts:{np.bincount(Y_ALL, minlength=TARGET_N_CLASSES).tolist()}")
print(f"Subjects:          {len(np.unique(SUBJECT_ID_ALL))}")
print(f"Window summary saved: {ARTIFACT_DIR / 'window_counts_by_subject.csv'}")
print(window_summary_df.head().to_string(index=False))

[2026-05-18 20:08:41] X_ALL shape:       (2000, 29, 537)
[2026-05-18 20:08:41] Y_ALL class counts:[1000, 1000]
[2026-05-18 20:08:41] Subjects:          50
[2026-05-18 20:08:41] Window summary saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/window_counts_by_subject.csv
[2026-05-18 20:08:41]  subject_id  n_windows class_counts  window_shape  resampled_samples_per_trial  crop_start_sample  crop_stop_sample  target_window_samples  effective_window_duration_s
          1         40     [20, 20] (40, 29, 537)                         1024                256               793                    537                     4.195312
          2         40     [20, 20] (40, 29, 537)                         1024                256               793                    537                     4.195312
          3         40     [20, 20] (40, 29, 537)                         10

## 3.5 Create Subject Datasets

In [9]:
class SubjectArrayDataset(Dataset):
    def __init__(self, X, y, subject_id):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subject_id = str(subject_id)

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        return self.X[idx], int(self.y[idx])

def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

SUBJECT_WINDOWS = {}
for sid in sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key):
    idx = np.where(SUBJECT_ID_ALL == sid)[0]
    SUBJECT_WINDOWS[str(sid)] = SubjectArrayDataset(X_ALL[idx], Y_ALL[idx], subject_id=sid)

print("Summarizing per-subject S-JEPA windows...")
for sid in sorted(SUBJECT_WINDOWS, key=_sort_subject_key):
    ds = SUBJECT_WINDOWS[sid]
    y = np.asarray([int(ds[i][1]) for i in range(len(ds))], dtype=np.int64)
    x0 = ds[0][0]
    counts = np.bincount(y, minlength=TARGET_N_CLASSES)
    print(f"  Subject {sid}: n_windows={len(ds)}, window_shape={tuple(x0.shape)}, class_counts={counts.tolist()}")

sample_x, sample_y = SUBJECT_WINDOWS[sorted(SUBJECT_WINDOWS, key=_sort_subject_key)[0]][0]
print(f"Sample window shape: {tuple(sample_x.shape)}")
print(f"Expected n_chans={len(CH_NAMES)}, n_times={WINDOW_SAMPLES}")

[2026-05-18 20:08:41] Summarizing per-subject S-JEPA windows...
[2026-05-18 20:08:41]   Subject 1: n_windows=40, window_shape=(29, 537), class_counts=[20, 20]
[2026-05-18 20:08:41]   Subject 2: n_windows=40, window_shape=(29, 537), class_counts=[20, 20]
[2026-05-18 20:08:41]   Subject 3: n_windows=40, window_shape=(29, 537), class_counts=[20, 20]
[2026-05-18 20:08:41]   Subject 4: n_windows=40, window_shape=(29, 537), class_counts=[20, 20]
[2026-05-18 20:08:41]   Subject 5: n_windows=40, window_shape=(29, 537), class_counts=[20, 20]
[2026-05-18 20:08:41]   Subject 6: n_windows=40, window_shape=(29, 537), class_counts=[20, 20]
[2026-05-18 20:08:41]   Subject 7: n_windows=40, window_shape=(29, 537), class_counts=[20, 20]
[2026-05-18 20:08:41]   Subject 8: n_windows=40, window_shape=(29, 537), class_counts=[20, 20]
[2026-05-18 20:08:41]   Subject 9: n_windows=40, window_shape=(29, 537), class_counts=[20, 20]
[2026-05-18 20:08:41]   Subject 10: n_windows=40, window_shape=(29, 537), class_c

# 4. Model

## 4.1 Build SignalJEPA PreLocal

In [10]:
def build_model():
    n_chans = len(CH_NAMES)
    n_times = WINDOW_SAMPLES
    mode = CONFIG["pretrained_mode"]
    common_kwargs = {
        "n_chans": n_chans,
        "chs_info": CHS_INFO,
        "n_times": n_times,
        "n_outputs": TARGET_N_CLASSES,
    }

    if mode == "from_pretrained":
        repo_id = CONFIG["pretrained_repo_id"]
        model = SignalJEPA_PreLocal.from_pretrained(
            repo_id,
            **common_kwargs,
            strict=False,
        )
        info = {"loading_path": "from_pretrained", "repo_id": repo_id, "mode": mode}
    elif mode == "random":
        model = SignalJEPA_PreLocal(**common_kwargs)
        info = {"loading_path": "random_initialization", "repo_id": None, "mode": mode}
    else:
        raise ValueError("CONFIG['pretrained_mode'] must be 'from_pretrained' or 'random'.")
    info["model_name"] = "SignalJEPA_PreLocal"
    return model, info

PRETRAINED_CHECKPOINT_INFO = {}

def initialize_model():
    return build_model()

# Verify once that the selected model builds.
test_model, test_info = build_model()
print("SignalJEPA_PreLocal instantiated successfully.")
print(f"  Loading mode:          {test_info['loading_path']}")
print(f"  Repo:                  {test_info.get('repo_id')}")
print(f"  Total parameters:      {sum(p.numel() for p in test_model.parameters()):,}")
print(f"  Trainable parameters:  {sum(p.numel() for p in test_model.parameters() if p.requires_grad):,}")
print(test_model)
del test_model

[2026-05-18 20:08:42] SignalJEPA_PreLocal instantiated successfully.
[2026-05-18 20:08:42]   Loading mode:          from_pretrained
[2026-05-18 20:08:42]   Repo:                  braindecode/signal-jepa_without-chans
[2026-05-18 20:08:42]   Total parameters:      16,010
[2026-05-18 20:08:42]   Trainable parameters:  16,010
[2026-05-18 20:08:42] ======================================================================================================================================================
Layer (type (var_name):depth-idx)                  Input Shape               Output Shape              Param #                   Kernel Shape
SignalJEPA_PreLocal (SignalJEPA_PreLocal)          [1, 29, 537]              [1, 2]                    --                        --
├─Sequential (spatial_conv): 1-1                   [1, 29, 537]              [1, 4, 537]               --                        --
│    └─Rearrange (0): 2-1                          [1, 29, 537]              [1, 1, 29, 537]    

## 4.2 Trainable Parameter Phases

In [11]:
NEW_LAYER_PREFIXES = ("spatial_conv.", "final_layer.")

def count_total_and_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def set_trainable_params_for_phase(model, phase):
    if phase not in ("new", "warmup", "full"):
        raise ValueError(f"Unsupported phase: {phase}")

    trainable_names = []
    if phase == "full":
        for _, param in model.named_parameters():
            param.requires_grad = True
        trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
        phase_groups = ["all_parameters"]
    else:
        for _, param in model.named_parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if any(name.startswith(prefix) for prefix in NEW_LAYER_PREFIXES):
                param.requires_grad = True
                trainable_names.append(name)
        phase_groups = list(NEW_LAYER_PREFIXES)

    total, trainable = count_total_and_trainable_params(model)
    if trainable == 0:
        raise RuntimeError(f"No trainable parameters found for phase={phase}.")
    return {
        "phase": phase,
        "trainable_groups": phase_groups,
        "total_params": int(total),
        "trainable_params": int(trainable),
        "trainable_ratio": float(trainable / total),
        "trainable_names": trainable_names,
    }

def assert_expected_trainable_scope(summary, phase):
    if phase == "full":
        return
    unexpected_names = [
        name for name in summary["trainable_names"]
        if not any(name.startswith(prefix) for prefix in NEW_LAYER_PREFIXES)
    ]
    if unexpected_names:
        raise RuntimeError(f"Unexpected trainable parameters for phase={phase}: {unexpected_names[:10]}")

def describe_trainable_params(summary, max_names=12):
    print(f"      Trainable groups: {summary['trainable_groups']}")
    print(f"      Trainable params: {summary['trainable_params']:,} / {summary['total_params']:,} ({summary['trainable_ratio']:.4%})")
    names = summary["trainable_names"]
    for name in names[:max_names]:
        print(f"        - {name}")
    if len(names) > max_names:
        print(f"        ... {len(names) - max_names} more")

def summarize_trainable_parameters(model):
    rows = []
    for name, param in model.named_parameters():
        rows.append({
            "name": name,
            "shape": list(param.shape),
            "n_params": int(param.numel()),
            "requires_grad": bool(param.requires_grad),
        })
    return rows

def count_trainable_from_rows(rows):
    return int(sum(row["n_params"] for row in rows if row["requires_grad"]))

## 4.3 PreLocal Spatial Convolution Weight Helpers

In [12]:
def _json_safe_float(value, decimals=8):
    if value is None:
        return None
    value = float(np.nan_to_num(value, nan=0.0, posinf=0.0, neginf=0.0))
    return round(value, decimals)

def _json_safe_float_list(values, decimals=8):
    arr = np.asarray(values, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.round(arr, decimals=decimals).tolist()

def get_model_module(model_or_clf):
    return model_or_clf.module_ if hasattr(model_or_clf, "module_") else model_or_clf

def _find_first_spatial_weight(model_or_clf):
    model = get_model_module(model_or_clf)
    if not hasattr(model, "spatial_conv"):
        return None, None

    candidates = []
    for name, param in model.named_parameters():
        if name.startswith("spatial_conv.") and name.endswith("weight") and param.ndim >= 2:
            candidates.append((name, param.detach().cpu().clone()))
    if candidates:
        candidates = sorted(candidates, key=lambda x: (0 if "spatial_conv.1.weight" in x[0] else 1, x[0]))
        return candidates[0]

    for name, module in model.spatial_conv.named_modules():
        if hasattr(module, "weight") and module.weight is not None and module.weight.ndim >= 2:
            return f"spatial_conv.{name}.weight", module.weight.detach().cpu().clone()
    return None, None

def _spatial_weight_to_channel_matrix(weight_tensor, n_chans):
    w = weight_tensor.detach().cpu().float().numpy()
    if w.ndim == 2:
        mat = w.copy()
    else:
        mat = w.reshape(w.shape[0], -1)

    if mat.shape[1] == n_chans:
        pass
    elif mat.shape[0] == n_chans:
        mat = mat.T
    elif mat.shape[1] % n_chans == 0:
        mat = mat.reshape(mat.shape[0], -1, n_chans).mean(axis=1)
    elif mat.size % n_chans == 0:
        mat = mat.reshape(-1, n_chans)
    else:
        raise ValueError(f"Cannot reshape spatial weight shape={w.shape} into n_chans={n_chans}.")
    return mat

def get_spatial_conv_weight_matrix(model_or_clf, ch_names):
    param_name, weight_tensor = _find_first_spatial_weight(model_or_clf)
    if weight_tensor is None:
        raise RuntimeError("Could not find spatial_conv weight on model.")
    weight_matrix = _spatial_weight_to_channel_matrix(weight_tensor, n_chans=len(ch_names))
    return weight_matrix.copy(), param_name

def compute_spatial_update_stats(initial_weight_matrix, final_weight_matrix, parameter_name):
    if initial_weight_matrix is None or final_weight_matrix is None:
        return None
    w0 = np.asarray(initial_weight_matrix, dtype=float)
    w1 = np.asarray(final_weight_matrix, dtype=float)
    if w0.shape != w1.shape:
        return {"available": False, "parameter_name": str(parameter_name), "reason": f"shape mismatch {w0.shape} vs {w1.shape}"}
    delta = w1 - w0
    init_l2 = float(np.linalg.norm(w0))
    return {
        "available": True,
        "parameter_name": str(parameter_name),
        "shape": list(w1.shape),
        "init_l2": _json_safe_float(init_l2),
        "final_l2": _json_safe_float(np.linalg.norm(w1)),
        "delta_l2": _json_safe_float(np.linalg.norm(delta)),
        "delta_max_abs": _json_safe_float(np.max(np.abs(delta))),
        "relative_delta": _json_safe_float(np.linalg.norm(delta) / max(init_l2, 1e-12)),
        "changed": bool(np.linalg.norm(delta) > 1e-10),
    }

def extract_spatial_conv_summary(model_or_clf, ch_names):
    try:
        weight_matrix, param_name = get_spatial_conv_weight_matrix(model_or_clf, ch_names)
    except Exception as exc:
        return {"available": False, "reason": str(exc)}

    abs_mean = np.mean(np.abs(weight_matrix), axis=0)
    signed_mean = np.mean(weight_matrix, axis=0)
    l2_scores = np.linalg.norm(weight_matrix, axis=0)

    return {
        "available": True,
        "parameter_name": param_name,
        "weight_shape": list(weight_matrix.shape),
        "channel_names": list(ch_names),
        "channel_abs_mean": _json_safe_float_list(abs_mean),
        "channel_signed_mean": _json_safe_float_list(signed_mean),
        "channel_l2": _json_safe_float_list(l2_scores),
        "global_abs_mean": _json_safe_float(np.mean(np.abs(weight_matrix))),
        "global_l2": _json_safe_float(np.linalg.norm(weight_matrix)),
    }

def summarize_spatial_conv_for_log(summary, top_k=5):
    if not summary or not summary.get("available"):
        return f"Spatial conv summary unavailable: {summary.get('reason') if summary else 'None'}"
    ch_names = summary["channel_names"]
    scores = np.asarray(summary["channel_abs_mean"], dtype=float)
    order = np.argsort(scores)[::-1][:top_k]
    top = ", ".join(f"{ch_names[i]}={scores[i]:.4g}" for i in order)
    return f"Spatial conv | {summary['parameter_name']} | top_abs_mean: {top}"

def summarize_spatial_update_for_log(stats):
    if not stats:
        return "Spatial update unavailable."
    if not stats.get("available"):
        return f"Spatial update unavailable: {stats.get('reason')}"
    return (
        f"Spatial update | delta_l2={stats['delta_l2']} | "
        f"relative_delta={stats['relative_delta']} | changed={stats['changed']}"
    )

# 5. Training

## 5.1 Build Classifier and Metrics Helpers

In [13]:
def get_targets(dataset):
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

def make_train_split():
    val_split = CONFIG["val_split"]
    if val_split is None or float(val_split) <= 0.0:
        return None
    return ValidSplit(cv=float(val_split), stratified=True, random_state=12)

def make_callbacks():
    train_split = make_train_split()
    patience = CONFIG["early_stopping_patience"]
    if train_split is None or patience is None or int(patience) <= 0:
        return []
    return [
        (
            "early_stopping",
            EarlyStopping(
                monitor="valid_loss",
                patience=int(patience),
                lower_is_better=True,
                load_best=True,
            ),
        )
    ]

def build_classifier(model, callbacks, max_epochs, fold_seed=None, warm_start=False):
    train_generator = None
    if fold_seed is not None:
        train_generator = torch.Generator()
        train_generator.manual_seed(fold_seed)

    clf_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "max_epochs": int(max_epochs),
        "device": DEVICE,
        "callbacks": callbacks,
        "train_split": make_train_split(),
        "classes": range(TARGET_N_CLASSES),
        "iterator_train__shuffle": True,
        "iterator_train__num_workers": 0,
        "iterator_valid__num_workers": 0,
        "optimizer": torch.optim.Adam,
        "warm_start": warm_start,
    }
    if CONFIG["learning_rate"] is not None:
        clf_kwargs["lr"] = CONFIG["learning_rate"]
    if train_generator is not None:
        clf_kwargs["iterator_train__generator"] = train_generator
    return EEGClassifier(model, **clf_kwargs)

def run_one_batch_finite_sanity_check(model, train_set):
    if len(train_set) == 0:
        raise RuntimeError("train_set is empty during sanity check.")
    batch_size = int(min(CONFIG["batch_size"], len(train_set)))
    sanity_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=False, num_workers=0)
    batch = next(iter(sanity_loader))
    x_batch = torch.as_tensor(batch[0]).float().to(DEVICE)
    y_batch = torch.as_tensor(batch[1]).long().to(DEVICE)
    was_training = model.training
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(x_batch)
        if not torch.isfinite(logits).all():
            raise RuntimeError("Non-finite logits detected.")
        loss = torch.nn.functional.cross_entropy(logits, y_batch)
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss detected.")
    if was_training:
        model.train()
    print("    Sanity check passed: finite logits/loss on one training batch.")

def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_pred = np.asarray(y_pred).astype(int).reshape(-1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

def compute_collapse_diagnostics(y_pred, n_classes):
    pred_hist = np.bincount(np.asarray(y_pred, dtype=int), minlength=n_classes)
    n_pred = int(pred_hist.sum())
    collapse_ratio = float(pred_hist.max() / n_pred) if n_pred else 0.0
    majority_class = int(pred_hist.argmax()) if n_pred else None
    threshold = float(CONFIG.get("collapse_threshold", 0.90))
    return {
        "prediction_histogram": pred_hist.tolist(),
        "collapse_ratio": _json_safe_float(collapse_ratio),
        "collapse_threshold": threshold,
        "collapse_flag": bool(collapse_ratio >= threshold),
        "majority_predicted_class": majority_class,
    }

def compute_prediction_probability_diagnostics(clf, test_set, y_pred, n_classes):
    if not bool(CONFIG.get("log_probability_diagnostics", True)):
        return None
    try:
        probs = np.asarray(clf.predict_proba(test_set), dtype=float)
    except Exception as exc:
        return {"available": False, "reason": f"predict_proba failed: {exc}"}

    if probs.ndim != 2 or probs.shape[0] != len(test_set):
        return {"available": False, "reason": f"Unexpected probability shape: {list(probs.shape)}"}
    if not np.isfinite(probs).all():
        return {"available": False, "reason": "Non-finite probabilities."}

    row_sums = probs.sum(axis=1, keepdims=True)
    if np.any(probs < 0) or not np.allclose(row_sums, 1.0, atol=1e-3):
        exp_probs = np.exp(probs - probs.max(axis=1, keepdims=True))
        probs = exp_probs / np.maximum(exp_probs.sum(axis=1, keepdims=True), 1e-12)

    eps = 1e-12
    confidence = probs.max(axis=1)
    entropy = -np.sum(probs * np.log(probs + eps), axis=1)
    normalized_entropy = entropy / np.log(max(probs.shape[1], 2))
    predicted_class_probability = probs[np.arange(len(probs)), np.asarray(y_pred, dtype=int)]

    return {
        "available": True,
        "probability_shape": list(probs.shape),
        "mean_probability_by_class": _json_safe_float_list(probs.mean(axis=0)),
        "std_probability_by_class": _json_safe_float_list(probs.std(axis=0)),
        "mean_confidence": _json_safe_float(confidence.mean()),
        "std_confidence": _json_safe_float(confidence.std()),
        "mean_prediction_entropy": _json_safe_float(entropy.mean()),
        "mean_normalized_prediction_entropy": _json_safe_float(normalized_entropy.mean()),
        "mean_predicted_class_probability": _json_safe_float(predicted_class_probability.mean()),
    }

## 5.2 Subject Cross-Validation Runner

In [14]:
def run_training_and_eval(train_set, test_set, fold_id, fold_label, n_total_folds=None):
    global PRETRAINED_CHECKPOINT_INFO

    if CONFIG["set_seed"]:
        fold_seed = BASE_SEED
        seed_everything(fold_seed)
    else:
        fold_seed = None

    y_train = get_targets(train_set)
    y_test = get_targets(test_set)
    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    strategy = CONFIG["strategy"]
    warmup_epochs = int(CONFIG["warmup_epochs"])
    model, pretrained_load_summary = initialize_model()
    PRETRAINED_CHECKPOINT_INFO = dict(pretrained_load_summary)

    total_folds_text = f"/{n_total_folds}" if n_total_folds is not None else ""
    print(f"\nFold {fold_id}{total_folds_text} | {fold_label}")
    print(f"    Train windows:           {len(train_set)} | counts={train_counts.tolist()}")
    print(f"    Test windows:            {len(test_set)} | counts={test_counts.tolist()}")
    print(f"    Downstream model:        SignalJEPA_PreLocal")
    print(f"    Fine-tune strategy:      {strategy}")
    print(f"    Pretrained loading path: {pretrained_load_summary['loading_path']}")
    print(f"    Pretrained repo:         {pretrained_load_summary.get('repo_id')}")

    initial_spatial_weight_matrix = None
    initial_spatial_param_name = None
    initial_spatial_summary = None
    if bool(CONFIG.get("extract_spatial_conv_weights", False)):
        initial_spatial_summary = extract_spatial_conv_summary(model, CH_NAMES)
        try:
            initial_spatial_weight_matrix, initial_spatial_param_name = get_spatial_conv_weight_matrix(model, CH_NAMES)
        except Exception as exc:
            print(f"    WARNING: Could not snapshot initial spatial_conv weights: {exc}")

    if strategy == "new":
        phase_1_summary = set_trainable_params_for_phase(model, "new")
        assert_expected_trainable_scope(phase_1_summary, "new")
        print("    Phase 1 (new):")
        describe_trainable_params(phase_1_summary)
        run_one_batch_finite_sanity_check(model, train_set)
        clf = build_classifier(model, callbacks=make_callbacks(), max_epochs=int(CONFIG["n_epochs"]), fold_seed=fold_seed, warm_start=False)
        phase_summaries = {"phase_1": phase_1_summary, "phase_2": None}
        clf.fit(train_set, y=y_train)
    elif strategy == "full":
        if warmup_epochs < 1:
            raise ValueError("For strategy='full', warmup_epochs must be >= 1.")
        phase_1_summary = set_trainable_params_for_phase(model, "warmup")
        assert_expected_trainable_scope(phase_1_summary, "warmup")
        print("    Phase 1 (warmup):")
        describe_trainable_params(phase_1_summary)
        run_one_batch_finite_sanity_check(model, train_set)

        clf = build_classifier(model, callbacks=[], max_epochs=warmup_epochs, fold_seed=fold_seed, warm_start=True)
        clf.fit(train_set, y=y_train)

        phase_2_summary = set_trainable_params_for_phase(clf.module_, "full")
        print("    Phase 2 (full):")
        describe_trainable_params(phase_2_summary)
        clf.initialize_optimizer()

        remaining_epochs = int(CONFIG["n_epochs"]) - warmup_epochs
        if remaining_epochs < 1:
            raise ValueError("CONFIG['n_epochs'] must be greater than warmup_epochs for strategy='full'.")
        clf.set_params(callbacks=make_callbacks(), max_epochs=remaining_epochs)
        clf.fit(train_set, y=y_train)
        phase_summaries = {"phase_1": phase_1_summary, "phase_2": phase_2_summary}
    else:
        raise ValueError("CONFIG['strategy'] must be 'new' or 'full'.")

    y_pred = clf.predict(test_set)
    metrics = compute_classification_metrics(y_test, y_pred)
    collapse_diagnostics = compute_collapse_diagnostics(y_pred, TARGET_N_CLASSES)
    probability_diagnostics = compute_prediction_probability_diagnostics(clf, test_set, y_pred, TARGET_N_CLASSES)

    spatial_summary = None
    spatial_update_stats = None
    if bool(CONFIG.get("extract_spatial_conv_weights", False)):
        spatial_summary = extract_spatial_conv_summary(clf.module_, CH_NAMES)
        if bool(CONFIG.get("log_spatial_update_stats", True)) and initial_spatial_weight_matrix is not None:
            try:
                final_spatial_weight_matrix, final_param_name = get_spatial_conv_weight_matrix(clf.module_, CH_NAMES)
                spatial_update_stats = compute_spatial_update_stats(
                    initial_weight_matrix=initial_spatial_weight_matrix,
                    final_weight_matrix=final_spatial_weight_matrix,
                    parameter_name=final_param_name or initial_spatial_param_name,
                )
            except Exception as exc:
                spatial_update_stats = {
                    "available": False,
                    "parameter_name": initial_spatial_param_name,
                    "reason": str(exc),
                }

    final_trainable_parameters = summarize_trainable_parameters(clf.module_)
    n_trainable_params_final = count_trainable_from_rows(final_trainable_parameters)

    stopped_epoch = int(clf.history[-1]["epoch"]) if len(clf.history) > 0 else 0
    valid_loss_curve = [(int(row["epoch"]), float(row["valid_loss"])) for row in clf.history if "valid_loss" in row]
    if valid_loss_curve:
        best_epoch, best_valid_loss = min(valid_loss_curve, key=lambda x: x[1])
    else:
        best_epoch, best_valid_loss = None, None

    cm = confusion_matrix(y_test, y_pred, labels=np.arange(TARGET_N_CLASSES)).tolist()
    pred_hist = collapse_diagnostics["prediction_histogram"]

    print(
        f"    Result | best_epoch={best_epoch} | stop={stopped_epoch} | "
        f"acc={metrics['accuracy']:.4f} | bal_acc={metrics['balanced_accuracy']:.4f} | "
        f"pred_hist={pred_hist} | collapse_ratio={collapse_diagnostics['collapse_ratio']:.4f} | "
        f"collapse={collapse_diagnostics['collapse_flag']}"
    )
    if spatial_summary is not None:
        print("    " + summarize_spatial_conv_for_log(spatial_summary))
    if spatial_update_stats is not None:
        print("    " + summarize_spatial_update_for_log(spatial_update_stats))

    return {
        "fold_id": int(fold_id),
        "fold_label": str(fold_label),
        "model_name": "SignalJEPA_PreLocal",
        "strategy": strategy,
        "warmup_epochs": warmup_epochs,
        "n_train": int(len(train_set)),
        "n_test": int(len(test_set)),
        "train_class_counts": train_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "pretrained_load": pretrained_load_summary,
        "phase_1_trainable_groups": phase_summaries["phase_1"]["trainable_groups"],
        "phase_1_trainable_params": int(phase_summaries["phase_1"]["trainable_params"]),
        "phase_1_trainable_names": phase_summaries["phase_1"]["trainable_names"],
        "phase_2_trainable_groups": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_groups"],
        "phase_2_trainable_params": None if phase_summaries["phase_2"] is None else int(phase_summaries["phase_2"]["trainable_params"]),
        "phase_2_trainable_names": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_names"],
        "final_trainable_parameters": final_trainable_parameters,
        "n_trainable_params_final": int(n_trainable_params_final),
        "best_epoch": best_epoch,
        "stopped_epoch": int(stopped_epoch),
        "best_valid_loss": best_valid_loss,
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
        "collapse_diagnostics": collapse_diagnostics,
        "probability_diagnostics": probability_diagnostics,
        "spatial_conv_initial": initial_spatial_summary,
        "spatial_conv": spatial_summary,
        "spatial_update_stats": spatial_update_stats,
    }

def make_fold_splits(y, n_folds, n_classes):
    indices = np.arange(len(y))
    counts = np.bincount(y, minlength=n_classes)
    min_class_count = counts.min()
    if min_class_count < n_folds:
        raise ValueError(f"Cannot use {n_folds} folds with class counts={counts.tolist()}.")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=12)
    return [
        {"fold_id": fold_id, "idx_train": train_idx, "idx_test": test_idx}
        for fold_id, (train_idx, test_idx) in enumerate(skf.split(indices, y), start=1)
    ]

def run_subject_cv(subject_id, subject_dataset, n_classes, cv_folds):
    y = get_targets(subject_dataset)
    counts = np.bincount(y, minlength=n_classes)
    print(f"\nSubject {subject_id}: {len(subject_dataset)} windows | class_counts={counts.tolist()}")
    folds = make_fold_splits(y, n_folds=cv_folds, n_classes=n_classes)
    results = []
    for fold in folds:
        train_set = Subset(subject_dataset, fold["idx_train"].tolist())
        test_set = Subset(subject_dataset, fold["idx_test"].tolist())
        fold_label = f"subject={subject_id}"
        result = run_training_and_eval(train_set, test_set, fold["fold_id"], fold_label, n_total_folds=cv_folds)
        result["subject_id"] = str(subject_id)
        results.append(result)

    acc_values = [r["accuracy"] for r in results if r["accuracy"] is not None]
    bal_acc_values = [r["balanced_accuracy"] for r in results if r["balanced_accuracy"] is not None]
    print(
        f"  Subject {subject_id} summary: "
        f"acc={np.mean(acc_values):.4f}±{np.std(acc_values):.4f}  "
        f"bal_acc={np.mean(bal_acc_values):.4f}±{np.std(bal_acc_values):.4f}"
    )
    return results

## 5.3 Run All Subjects

In [15]:
print("=" * 70)
print("STARTING WITHIN-SUBJECT CROSS-VALIDATION")
print("=" * 70)
print(f"Dataset:       {CONFIG['dataset_name']}")
print(f"Subjects:      {list(SUBJECT_WINDOWS.keys())}")
print(f"Model:         SignalJEPA_PreLocal")
print(f"Pretrained:    {CONFIG['pretrained_mode']}")
print(f"Strategy:      {CONFIG['strategy']}")
print(f"CV folds:      {CONFIG['cv_folds']}")
print(f"Val split:     {CONFIG['val_split']}")
print(f"Max epochs:    {CONFIG['n_epochs']}")
print(f"Device:        {DEVICE}")
print("=" * 70)

FOLD_RESULTS = []
for sid, subject_ds in SUBJECT_WINDOWS.items():
    FOLD_RESULTS.extend(run_subject_cv(sid, subject_ds, TARGET_N_CLASSES, CONFIG["cv_folds"]))

print(f"\nTotal folds completed: {len(FOLD_RESULTS)}")

[2026-05-18 20:08:42] ======================================================================
[2026-05-18 20:08:42] STARTING WITHIN-SUBJECT CROSS-VALIDATION
[2026-05-18 20:08:42] ======================================================================
[2026-05-18 20:08:42] Dataset:       Liu2024_SourceMAT
[2026-05-18 20:08:42] Subjects:      ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50']
[2026-05-18 20:08:42] Model:         SignalJEPA_PreLocal
[2026-05-18 20:08:42] Pretrained:    from_pretrained
[2026-05-18 20:08:42] Strategy:      new
[2026-05-18 20:08:42] CV folds:      5
[2026-05-18 20:08:42] Val split:     0.2
[2026-05-18 20:08:42] Max epochs:    50
[2026-05-18 20:08:42] Device:        mps
[2026-05-18 20:08:42] =================================

# 6. Results

## 6.1 Aggregate Metrics

In [16]:
def aggregate_results(fold_results):
    grouped = {}
    for result in fold_results:
        sid = result.get("subject_id", "global")
        grouped.setdefault(sid, {"accuracies": [], "balanced_accuracies": []})
        grouped[sid]["accuracies"].append(result.get("accuracy"))
        grouped[sid]["balanced_accuracies"].append(result.get("balanced_accuracy"))

    for sid, m in grouped.items():
        acc_values = [v for v in m["accuracies"] if v is not None]
        bal_values = [v for v in m["balanced_accuracies"] if v is not None]
        m["mean_accuracy"] = float(np.mean(acc_values)) if acc_values else None
        m["std_accuracy"] = float(np.std(acc_values)) if acc_values else None
        m["mean_balanced_accuracy"] = float(np.mean(bal_values)) if bal_values else None
        m["std_balanced_accuracy"] = float(np.std(bal_values)) if bal_values else None

    all_accs = [r["accuracy"] for r in fold_results if r.get("accuracy") is not None]
    all_bals = [r["balanced_accuracy"] for r in fold_results if r.get("balanced_accuracy") is not None]
    global_metrics = {
        "mean_accuracy": float(np.mean(all_accs)) if all_accs else None,
        "std_accuracy": float(np.std(all_accs)) if all_accs else None,
        "mean_balanced_accuracy": float(np.mean(all_bals)) if all_bals else None,
        "std_balanced_accuracy": float(np.std(all_bals)) if all_bals else None,
        "n_subjects": len(grouped),
        "n_folds_total": len(fold_results),
    }
    return grouped, global_metrics

SUBJECT_METRICS, GLOBAL_METRICS = aggregate_results(FOLD_RESULTS)

print("=" * 70)
print("AGGREGATED RESULTS")
print("=" * 70)
for sid, m in sorted(SUBJECT_METRICS.items(), key=lambda x: _sort_subject_key(x[0])):
    acc_str = f"{m['mean_accuracy']:.4f}±{m['std_accuracy']:.4f}" if m["mean_accuracy"] is not None else "N/A"
    bal_str = f"{m['mean_balanced_accuracy']:.4f}±{m['std_balanced_accuracy']:.4f}" if m["mean_balanced_accuracy"] is not None else "N/A"
    print(f"  {sid}: acc={acc_str}  bal_acc={bal_str}")

print("-" * 70)
print(
    f"  OVERALL: acc={GLOBAL_METRICS['mean_accuracy']:.4f}±{GLOBAL_METRICS['std_accuracy']:.4f}  "
    f"bal_acc={GLOBAL_METRICS['mean_balanced_accuracy']:.4f}±{GLOBAL_METRICS['std_balanced_accuracy']:.4f}"
)
print("=" * 70)

[2026-05-18 20:12:25] ======================================================================
[2026-05-18 20:12:25] AGGREGATED RESULTS
[2026-05-18 20:12:25] ======================================================================
[2026-05-18 20:12:25]   1: acc=0.4500±0.1500  bal_acc=0.4500±0.1500
[2026-05-18 20:12:25]   2: acc=0.4750±0.0500  bal_acc=0.4750±0.0500
[2026-05-18 20:12:25]   3: acc=0.5000±0.1369  bal_acc=0.5000±0.1369
[2026-05-18 20:12:25]   4: acc=0.6500±0.1225  bal_acc=0.6500±0.1225
[2026-05-18 20:12:25]   5: acc=0.5500±0.1275  bal_acc=0.5500±0.1275
[2026-05-18 20:12:25]   6: acc=0.5750±0.1500  bal_acc=0.5750±0.1500
[2026-05-18 20:12:25]   7: acc=0.4000±0.0935  bal_acc=0.4000±0.0935
[2026-05-18 20:12:25]   8: acc=0.5250±0.1837  bal_acc=0.5250±0.1837
[2026-05-18 20:12:25]   9: acc=0.6000±0.1225  bal_acc=0.6000±0.1225
[2026-05-18 20:12:25]   10: acc=0.4000±0.0935  bal_acc=0.4000±0.0935
[2026-05-18 20:12:25]   11: acc=0.4500±0.1696  bal_acc=0.4500±0.1696
[2026-05-18 20:12:25]  

## 6.2 Experiment Summary

In [17]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:                 {RUN_ID}")
print(f"Dataset:                {CONFIG['dataset_name']}")
print(f"Source extract dir:     {SOURCE_EXTRACT_DIR}")
print(f"Model:                  SignalJEPA_PreLocal")
print(f"Pretrained mode:        {CONFIG['pretrained_mode']}")
print(f"Pretrained repo:        {CONFIG['pretrained_repo_id']}")
print(f"Strategy:               {CONFIG['strategy']}")
print(f"Window:                 {TARGET_TRIAL_DURATION_S:.6f}s / {WINDOW_SAMPLES} samples")
print(f"Window start:           {CONFIG['mi_window_start_s']}s")
print(f"Channels:               {len(CH_NAMES)}")
print(f"Artifacts:              {ARTIFACT_DIR}")
print(f"Mean Accuracy:          {GLOBAL_METRICS['mean_accuracy']:.4f} ± {GLOBAL_METRICS['std_accuracy']:.4f}")
print(f"Mean Balanced Accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']:.4f} ± {GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)

[2026-05-18 20:12:25] 
[2026-05-18 20:12:25] EXPERIMENT SUMMARY
[2026-05-18 20:12:25] ======================================================================
[2026-05-18 20:12:25] Run ID:                 20260518_2008_8d0bbab1
[2026-05-18 20:12:25] Dataset:                Liu2024_SourceMAT
[2026-05-18 20:12:25] Source extract dir:     /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata
[2026-05-18 20:12:25] Model:                  SignalJEPA_PreLocal
[2026-05-18 20:12:25] Pretrained mode:        from_pretrained
[2026-05-18 20:12:25] Pretrained repo:        braindecode/signal-jepa_without-chans
[2026-05-18 20:12:25] Strategy:               new
[2026-05-18 20:12:25] Window:                 4.195312s / 537 samples
[2026-05-18 20:12:25] Window start:           2.0s
[2026-05-18 20:12:25] Channels:               29
[2026-05-18 20:12:25] Artifacts:              /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_J

## 6.3 Spatial Convolution Weight Analysis

In [18]:
def collect_spatial_conv_records(fold_results):
    records = []
    for result in fold_results:
        spatial = result.get("spatial_conv")
        if not spatial or not spatial.get("available", False):
            continue

        ch_names = spatial["channel_names"]
        abs_scores = np.asarray(spatial["channel_abs_mean"], dtype=float)
        signed_scores = np.asarray(spatial["channel_signed_mean"], dtype=float)
        l2_scores = np.asarray(spatial["channel_l2"], dtype=float)
        collapse = result.get("collapse_diagnostics") or {}
        update_stats = result.get("spatial_update_stats") or {}

        for ch_idx, ch_name in enumerate(ch_names):
            records.append({
                "dataset_name": CONFIG["dataset_name"],
                "subject_id": result.get("subject_id"),
                "fold_id": result.get("fold_id"),
                "channel_index": ch_idx,
                "channel_name": ch_name,
                "abs_mean_weight": float(abs_scores[ch_idx]),
                "signed_mean_weight": float(signed_scores[ch_idx]),
                "l2_weight": float(l2_scores[ch_idx]),
                "accuracy": result.get("accuracy"),
                "balanced_accuracy": result.get("balanced_accuracy"),
                "prediction_histogram": result.get("prediction_histogram"),
                "collapse_ratio": collapse.get("collapse_ratio"),
                "collapse_flag": collapse.get("collapse_flag"),
                "spatial_delta_l2": update_stats.get("delta_l2"),
                "spatial_delta_max_abs": update_stats.get("delta_max_abs"),
                "spatial_relative_delta": update_stats.get("relative_delta"),
                "spatial_changed": update_stats.get("changed"),
            })
    return records

def collect_fold_diagnostic_records(fold_results):
    rows = []
    for result in fold_results:
        collapse = result.get("collapse_diagnostics") or {}
        update_stats = result.get("spatial_update_stats") or {}
        prob = result.get("probability_diagnostics") or {}
        rows.append({
            "dataset_name": CONFIG["dataset_name"],
            "subject_id": result.get("subject_id"),
            "fold_id": result.get("fold_id"),
            "accuracy": result.get("accuracy"),
            "balanced_accuracy": result.get("balanced_accuracy"),
            "prediction_histogram": result.get("prediction_histogram"),
            "collapse_ratio": collapse.get("collapse_ratio"),
            "collapse_flag": collapse.get("collapse_flag"),
            "majority_predicted_class": collapse.get("majority_predicted_class"),
            "spatial_delta_l2": update_stats.get("delta_l2"),
            "spatial_delta_max_abs": update_stats.get("delta_max_abs"),
            "spatial_relative_delta": update_stats.get("relative_delta"),
            "spatial_changed": update_stats.get("changed"),
            "mean_confidence": prob.get("mean_confidence") if prob.get("available") else None,
            "mean_normalized_prediction_entropy": prob.get("mean_normalized_prediction_entropy") if prob.get("available") else None,
            "n_trainable_params_final": result.get("n_trainable_params_final"),
        })
    return rows

SPATIAL_CONV_RECORDS = collect_spatial_conv_records(FOLD_RESULTS)
FOLD_DIAGNOSTIC_RECORDS = collect_fold_diagnostic_records(FOLD_RESULTS)

spatial_dir = ARTIFACT_DIR / "spatial_conv_analysis"
spatial_dir.mkdir(parents=True, exist_ok=True)

fold_diag_df = pd.DataFrame(FOLD_DIAGNOSTIC_RECORDS)
fold_diag_path = spatial_dir / "fold_diagnostics.csv"
fold_diag_df.to_csv(fold_diag_path, index=False)
print(f"Fold diagnostics saved to: {fold_diag_path}")

if len(SPATIAL_CONV_RECORDS) == 0:
    print("Spatial convolution analysis skipped: no available spatial_conv records were extracted.")
else:
    spatial_df = pd.DataFrame(SPATIAL_CONV_RECORDS)
    spatial_long_path = spatial_dir / "spatial_channel_importance_long.csv"
    spatial_df.to_csv(spatial_long_path, index=False)

    global_df = (
        spatial_df
        .groupby("channel_name", as_index=False)
        .agg(
            mean_abs_weight=("abs_mean_weight", "mean"),
            mean_l2_weight=("l2_weight", "mean"),
            mean_signed_weight=("signed_mean_weight", "mean"),
            n_records=("abs_mean_weight", "count"),
        )
        .sort_values("mean_abs_weight", ascending=False)
    )
    global_path = spatial_dir / "spatial_channel_importance_global.csv"
    global_df.to_csv(global_path, index=False)

    print(f"Spatial long records saved to: {spatial_long_path}")
    print(f"Global channel importance saved to: {global_path}")
    print("Top channels by mean_abs_weight:")
    print(global_df.head(10).to_string(index=False))

[2026-05-18 20:12:25] Fold diagnostics saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/spatial_conv_analysis/fold_diagnostics.csv
[2026-05-18 20:12:25] Spatial long records saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/spatial_conv_analysis/spatial_channel_importance_long.csv
[2026-05-18 20:12:25] Global channel importance saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/spatial_conv_analysis/spatial_channel_importance_global.csv
[2026-05-18 20:12:25] Top channels by mean_abs_weight:
[2026-05-18 20:12:25] channel_name  mean_abs_weight  mean_l2_weight  mean_signed_weight  n_records
         FC4   

## 6.4 Save Artifacts

In [19]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, "w") as f:
    json.dump(FOLD_RESULTS, f, indent=2)

subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, "w") as f:
    json.dump(SUBJECT_METRICS, f, indent=2)

global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, "w") as f:
    json.dump(GLOBAL_METRICS, f, indent=2)

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "dataset_name": CONFIG["dataset_name"],
    "source": "original Figshare sourcedata .mat files",
    "source_extract_dir": str(SOURCE_EXTRACT_DIR),
    "labels_to_keep": list(CONFIG["labels_to_keep"]),
    "subjects": [str(s) for s in SUBJECTS],
    "excluded_subjects": list(CONFIG["exclude_subjects"]),
    "source_sfreq": CONFIG["source_sfreq"],
    "target_sfreq": CONFIG["sfreq"],
    "preprocessing_order": [
        "load source .mat rawdata trials x channels x samples",
        "select EEG channels; optionally drop CPz source reference, EOG, and marker",
        "concatenate trials per subject into RawArray",
        "average reference" if CONFIG["average_reference_before_resample_filter"] else "average reference after filter",
        "resample to 128 Hz",
        "bandpass 0.5-40 Hz",
        "reshape back to trials",
        "crop fixed 537-sample MI window starting at 2.0s",
    ],
    "channel_names": list(CH_NAMES),
    "target_window_duration_s": TARGET_TRIAL_DURATION_S,
    "window_samples": WINDOW_SAMPLES,
    "mi_window_start_s": CONFIG["mi_window_start_s"],
    "mi_window_start_sample": MI_WINDOW_START_SAMPLE,
    "mi_window_stop_sample": MI_WINDOW_STOP_SAMPLE,
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": CONFIG["pretrained_mode"],
    "pretrained_repo_id": CONFIG["pretrained_repo_id"],
    "strategy": CONFIG["strategy"],
    "warmup_epochs": int(CONFIG["warmup_epochs"]),
    "pretrained_checkpoint_info": PRETRAINED_CHECKPOINT_INFO,
    "cv_seed": BASE_SEED,
    "cv_folds": CONFIG["cv_folds"],
    "val_split": CONFIG["val_split"],
    "global_metrics": GLOBAL_METRICS,
    "extract_spatial_conv_weights": bool(CONFIG.get("extract_spatial_conv_weights", False)),
    "collapse_threshold": float(CONFIG.get("collapse_threshold", 0.90)),
    "log_spatial_update_stats": bool(CONFIG.get("log_spatial_update_stats", True)),
    "log_probability_diagnostics": bool(CONFIG.get("log_probability_diagnostics", True)),
}

run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2)

print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")

_LOG_FILE_HANDLE.close()

[2026-05-18 20:12:25] CV results saved to:      /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/cv_results.json
[2026-05-18 20:12:25] Subject metrics saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/subject_metrics.json
[2026-05-18 20:12:25] Global metrics saved to:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/global_metrics.json
[2026-05-18 20:12:25] Run metadata saved to:    /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/artifacts/liu2024-source-mat-sjepa-prelocal/Liu2024_SourceMAT/20260518_2008_8d0bbab1/run_metadata.json
[2026-05-18 20:12:25] 
All artifacts in: /Users/vadim